# Global snowmelt runoff onset icechunk store creation (v10+)

Creates and initializes the **icechunk** repository holding the Zarr v3 global output store that per-tile-per-water-year processing jobs commit into. Replaces `create_zarr_store.ipynb` (which pre-allocated the plain Zarr v2 store used through v9 -- that store stays frozen as published).

## Store design

- **Same five variables as v9** (`runoff_onset`, `runoff_onset_median`, `runoff_onset_mad`, `temporal_resolution`, `temporal_resolution_median`), int16 on disk, -9999 nodata, 0.1-day scaling for MAD/temporal resolution.
- **Zarr v3 sharding**: shards of (1 water_year, 2048, 2048) = exactly one tile x one water year, so each processing commit writes whole shards and concurrent tile jobs never touch the same object. Inner chunks of (1, 256, 256) keep point/station reads small (256 vs 512 benchmarked on Azure 2026-07: point-read p90 24% better at 256, everything else within noise).
- **Metadata-only init**: the template is lazy dask; only Zarr metadata + coordinates are written (`compute=False`, `write_empty_chunks=False`). Unprocessed regions simply have no chunks.
- **Manifest splitting** (persisted via `repo.save_config()`): one manifest per water year per array, so each of the ~50k processing commits rewrites only a small manifest instead of the full chunk-reference list. See [icechunk performance guide](https://icechunk.io/en/latest/guides/performance/).
- **Status = commit history**: processing state is derived from structured commit metadata (`global_snowmelt_runoff_onset/status.py`); there is no CSV/status file to maintain.


In [ ]:
import icechunk
import xarray as xr

from global_snowmelt_runoff_onset.config import Config
from global_snowmelt_runoff_onset import store
from global_snowmelt_runoff_onset.provenance import collect_provenance

config = Config('config/global_config_v10.txt')
print(f'output repo: {config.global_runoff_icechunk_azure_prefix}')
print(f'shard: (1, {config.spatial_chunk_dim_zarr_output}, {config.spatial_chunk_dim_zarr_output}), '
      f'inner chunks: (1, {config.inner_chunk_dim}, {config.inner_chunk_dim})')

## Preview the template

Lazy dataset -- nothing is computed or uploaded here.


In [ ]:
template_ds, encoding = store.build_template(config)
display(template_ds)
encoding['runoff_onset']

## Create the repository and write the template

`Repository.create` fails if a repo already exists at the prefix -- it will not silently overwrite. To truly start over, delete the prefix first (deliberately manual):

```python
# import adlfs
# fs = adlfs.AzureBlobFileSystem(account_name=config.azure_storage_account, credential=config.sas_token)
# fs.rm(config.global_runoff_icechunk_azure_prefix, recursive=True)
```


In [ ]:
repo = config.create_output_repo()  # persists manifest-splitting + retry config on-disk
snapshot_id = store.initialize_store(repo, config, extra_metadata={'provenance': collect_provenance()})
print(f'initialized: {snapshot_id}')

## Verify


In [ ]:
session = repo.readonly_session('main')
global_zarr_ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all')
display(global_zarr_ds)
for var in global_zarr_ds.data_vars:
    print(var, global_zarr_ds[var].encoding.get('shards'), global_zarr_ds[var].encoding.get('chunks'),
          global_zarr_ds[var].encoding.get('dtype'))

## Notes

- **Appending a future water year** (e.g. WY2026): update `WY_end` in a new config, `to_zarr(..., append_dim='water_year')` a metadata-only slice (or resize via zarr), then dispatch tile jobs with `--water-years 2026`; composite commits refresh per-tile as those jobs run (staleness is tracked automatically).
- **At publication**: `repo.expire_snapshots(...)` + `repo.garbage_collect(...)` to compact the ~50k processing commits, then `repo.create_tag('v10.0', snapshot_id=...)` so readers can pin the released version.
